# Support Vector Regression

_Classical ML_

**Fit a line inside a tube. Ignore points the tube already covers.**

Support Vector Regression (SVR) fits a function but tolerates small errors.

     It draws a tube of width $\pm\varepsilon$ around the fitted line.

---

This notebook builds Support Vector Regression up one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

We fit SVR on a clean toy signal first: a noisy sine wave. Three steps — (1) make the data, (2) fit the SVR, (3) read off how many points the tube already covers.

### Step 1 — Make a noisy sine wave

We draw 120 input points in `[0, 6]`, sort them, and set the target to `sin(x)` plus a little Gaussian noise. The sine gives a curved signal that a straight line could not capture, which is why we will reach for an RBF kernel in the next step.

In [ ]:
import numpy as np
from sklearn.svm import SVR

rng = np.random.default_rng(0)

# 120 sorted inputs, target is sin(x) plus small noise.
X = np.sort(rng.uniform(0, 6, size=(120, 1)), axis=0)
y = np.sin(X[:, 0]) + rng.normal(0, 0.1, size=120)

### Step 2 — Fit the SVR with an epsilon-tube

SVR fits a curve but draws a tube of width `epsilon = 0.1` around it and **ignores any point that already falls inside the tube** — only points on or outside the tube boundary affect the fit. The `rbf` kernel lets the curve bend to follow the sine, and `C` controls how hard we push to keep points inside the tube.

In [ ]:
# epsilon sets the tube half-width; rbf lets the fit curve.
svr = SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale").fit(X, y)

# Predict on the same points to measure the fit.
pred = svr.predict(X)

### Step 3 — Read off the support vectors

The points that lie on or outside the tube are the **support vectors** — they alone define the fit. The fraction *inside* the tube is the fraction SVR could safely ignore. We also report R^2 (how well the curve explains the data) and the largest residual.

In [ ]:
print("R^2 on training data :", round(svr.score(X, y), 3))
print("support vectors      :", len(svr.support_), "of", len(X))
print("fraction inside tube :",
      round(1 - len(svr.support_) / len(X), 3))
print("max residual         :", round(np.abs(y - pred).max(), 3))

## Visualize the data & results

_On real diabetes data, where does SVR fit BMI against disease progression?_ We move to 442 real patients in three steps: (1) pull out the BMI feature and sort it, (2) fit SVR over a smooth grid, (3) plot the fit together with its epsilon-tube.

### Step 1 — Pull out the BMI feature and sort it

From the diabetes dataset we take a single feature — BMI (already standardised) — as the input and one-year disease progression as the target. Sorting the points by BMI lets us draw a clean fitted curve from left to right later.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.svm import SVR

db = load_diabetes()                     # 442 real patients
x = db.data[:, 2]                        # BMI feature (standardized)
y = db.target                            # disease progression after one year

# Sort by BMI so the fitted curve plots left to right.
order = np.argsort(x)
x = x[order]
y = y[order]

### Step 2 — Fit SVR and predict over a smooth grid

Here the target ranges in the hundreds, so we widen the tube to `epsilon = 25` and raise `C` to 1000 to fit the trend firmly. We then predict over 200 evenly spaced BMI values, giving a smooth curve to draw rather than a jagged point-to-point line.

In [ ]:
eps = 25.0

# A wide tube and large C to fit this larger-scale target.
svr = SVR(kernel="rbf", C=1000.0, epsilon=eps, gamma="scale").fit(x.reshape(-1, 1), y)

# Evaluate the fit on a smooth grid of BMI values.
grid = np.linspace(x.min(), x.max(), 200).reshape(-1, 1)
fit = svr.predict(grid)

### Step 3 — Plot the patients, the fit, and the tube

We scatter the real patients, draw the SVR fit in green, and add the two orange tube boundaries at `fit ± epsilon`. Points falling between the orange lines are the ones SVR treats as 'close enough' and ignores when fitting.

In [ ]:
plt.scatter(x, y, c="#4ea1ff", s=12, label="patients")
plt.plot(grid[:, 0], fit, c="#7ee787", label="SVR fit")
plt.plot(grid[:, 0], fit + eps, c="#ffb454", label="tube upper")
plt.plot(grid[:, 0], fit - eps, c="#ffb454", label="tube lower")
plt.title("SVR fit of disease progression vs BMI with the epsilon-tube")
plt.xlabel("BMI (standardized)")
plt.ylabel("disease progression")
plt.legend()
plt.show()